# Despacho de 3 centrales con límite de potencia y referencia de presión

**Idea física**
- Cada planta tiene dos bombas (mismo índice `i`): `circ_pump_pressure[i]` (planta **slack**: fija la presión, su potencia es lo que salga) y `circ_pump_mass[i]` (planta con **caudal impuesto**: se ajusta `mdot` hasta entregar una potencia objetivo). Se activa una, la otra, o ninguna (planta apagada).
- **Orden de mérito**: se prueba con la planta 1 sola; si pasa de su `Q_max`, se fija a `Q_max` y entra la 2; si la 2 pasa de su `Q_max`, entra la 3, etc.
- **La planta 1 es siempre la referencia de presión** (3 bar ida / 2.5 bar retorno). Cuando está saturada se queda a `Q_max` *y sigue siendo slack*: la planta marginal ajusta su caudal hasta que la planta 1 entregue exactamente `Q_max`.
- La potencia de cada planta se calcula siempre igual: `mdot · cp_medio · (T_ida − T_retorno)`.
- Si ni con todas las plantas a su máximo se cubre la demanda, el código lo dice (`INFACTIBLE`) y no lo disfraza.

In [462]:
# import numpy as np
import numpy as np
import pandapipes as pp
from pandapipes.pf.pipeflow_setup import get_fluid
# 1. Crear red vacía
net = pp.create_empty_network(fluid="water")

# =============================================================================
# NUDOS (JUNCTIONS)
# =============================================================================
# Nota: Las temperaturas en pandapipes se introducen en Kelvin (K = °C + 273.15)
# Ida 
j_fuente_ida = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Fuente_Ida")
j_fuente_2_ida = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Fuente_2_Ida")
j_fuente_3_ida = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Fuente_3_Ida")
j_nodo_1=pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Nodo1_ida")
j_nodo_2=pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Nodo2_ida")
j_nodo_3=pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Nodo3_ida")
j_nodo_4=pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Nodo4_ida")
j_nodo_5=pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Nodo5_ida")
j_nodo_6=pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Nodo6_ida")
j_cons_ida_1 = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Consumidor1_Ida")
j_cons_ida_2 = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Consumidor2_Ida")
j_cons_ida_3 = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Consumidor3_Ida")
j_cons_ida_4 = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Consumidor4_Ida")
j_cons_ida_5 = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Consumidor5_Ida")
j_cons_ida_6 = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Consumidor6_Ida")
j_cons_ida_7 = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Consumidor7_Ida")
j_cons_ida_8 = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Consumidor8_Ida")
j_cons_ida_9 = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Consumidor9_Ida")
#Retorno 
j_fuente_ret = pp.create_junction(net, pn_bar=5.0, tfluid_k=305, name="Fuente_Retorno")
j_fuente_2_ret = pp.create_junction(net, pn_bar=5.0, tfluid_k=305, name="Fuente_2_Retorno")
j_fuente_3_ret = pp.create_junction(net, pn_bar=5.0, tfluid_k=305, name="Fuente_3_Retorno")
j_nodo_1_ret=pp.create_junction(net, pn_bar=5.0, tfluid_k=305, name="Nodo1_ret")
j_nodo_2_ret=pp.create_junction(net, pn_bar=5.0, tfluid_k=305, name="Nodo2_ret")
j_nodo_3_ret=pp.create_junction(net, pn_bar=5.0, tfluid_k=305, name="Nodo3_ret")
j_nodo_4_ret=pp.create_junction(net, pn_bar=5.0, tfluid_k=305, name="Nodo4_ret")
j_nodo_5_ret=pp.create_junction(net, pn_bar=5.0, tfluid_k=305, name="Nodo5_ret")
j_nodo_6_ret=pp.create_junction(net, pn_bar=5.0, tfluid_k=305, name="Nodo6_ret")
j_cons_ret_1 = pp.create_junction(net, pn_bar=5.0, tfluid_k=305, name="Consumidor1_Retorno")
j_cons_ret_2 = pp.create_junction(net, pn_bar=5.0, tfluid_k=305, name="Consumidor2_Retorno")
j_cons_ret_3 = pp.create_junction(net, pn_bar=5.0, tfluid_k=305, name="Consumidor3_Retorno")
j_cons_ret_4 = pp.create_junction(net, pn_bar=5.0, tfluid_k=305, name="Consumidor4_Retorno")
j_cons_ret_5 = pp.create_junction(net, pn_bar=5.0, tfluid_k=305, name="Consumidor5_Retorno")
j_cons_ret_6 = pp.create_junction(net, pn_bar=5.0, tfluid_k=305, name="Consumidor6_Retorno")
j_cons_ret_7 = pp.create_junction(net, pn_bar=5.0, tfluid_k=305, name="Consumidor7_Retorno")
j_cons_ret_8 = pp.create_junction(net, pn_bar=5.0, tfluid_k=305, name="Consumidor8_Retorno")
j_cons_ret_9 = pp.create_junction(net, pn_bar=5.0, tfluid_k=305, name="Consumidor9_Retorno")



In [463]:
# Parche de pandapipes (el tuyo, sin cambios)
import numpy as np
from pandapipes.component_models.abstract_models.circulation_pump import CirculationPump
from pandapipes.idx_branch import D, AREA, TOUTINIT
from pandapipes.idx_node import TINIT
from pandapipes.pf.pipeflow_setup import get_lookup

def _create_pit_branch_entries_fixed(cls, net, branch_pit):
    node_pit = net["_pit"]["node"]
    junction_idx_lookups = get_lookup(net, "node", "index")[
        cls.get_connected_node_type().table_name()]

    # FIX: usar la tabla COMPLETA, sin filtrar por in_service,
    # para que coincida con el tamaño de circ_pump_pit (que sí incluye
    # todas las filas, activas o no).
    circ_pump_tbl = net[cls.table_name()]
    circ_pump_pit = super(CirculationPump, cls).create_pit_branch_entries(net, branch_pit)
    circ_pump_pit[:, D] = 0.1
    circ_pump_pit[:, AREA] = circ_pump_pit[:, D] ** 2 * np.pi / 4

    types = circ_pump_tbl.type.values
    mask_t = np.isin(types, ["pt", "t"])
    juncts = circ_pump_tbl[cls.from_to_node_cols()[1]].values
    to_nodes = junction_idx_lookups[juncts]
    circ_pump_pit[mask_t, TOUTINIT] = circ_pump_tbl.t_flow_k.values[mask_t]
    circ_pump_pit[~mask_t, TOUTINIT] = node_pit[to_nodes[~mask_t], TINIT]
    return circ_pump_pit

CirculationPump.create_pit_branch_entries = classmethod(_create_pit_branch_entries_fixed)

In [464]:
# =============================================================================
# TUBERÍAS (PIPES) - IDA Y RETORNO SIN PÉRDIDAS TÉRMICAS (k_w = 0)
# =============================================================================

#Plant-Nodo
#Planta 1
pipe_ida_1 = pp.create_pipe_from_parameters(
    net, from_junction=j_fuente_ida, to_junction=j_nodo_1,
    length_km=0.0374, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15), text_k=273.2, name="Tubo_Ida_Plant_nodo_1",k_mm=0.1
)

pipe_retorno_1= pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_1_ret, to_junction=j_fuente_ret,
    length_km=0.0374, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15),  text_k=273.2, name="Tubo_RetornoPlant_nodo_1",k_mm=0.1
)
#Planta 2
pipe_ida_2 = pp.create_pipe_from_parameters(
    net, from_junction=j_fuente_2_ida, to_junction=j_nodo_3,
    length_km=0.0265, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15),  text_k=273.2, name="Tubo_Ida_Plant_2_nodo_3",k_mm=0.1
)

pipe_retorno_2= pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_3_ret, to_junction=j_fuente_2_ret,
    length_km=0.0265, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15),  text_k=273.2, name="Tubo_RetornoPlant_2_nodo_3",k_mm=0.1
)
#Planta 3
pipe_ida_3 = pp.create_pipe_from_parameters(
    net, from_junction=j_fuente_3_ida, to_junction=j_nodo_6,
    length_km=0.0624, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15),text_k=273.2, name="Tubo_Ida_Plant_3_nodo_6",k_mm=0.1)

pipe_retorno_3= pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_6_ret, to_junction=j_fuente_3_ret,
    length_km=0.0624, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15),  text_k=273.2, name="Tubo_RetornoPlant_3_nodo_6",k_mm=0.1
)
#Nodo-nodo
#1-2
pipe_ida_4 = pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_1, to_junction=j_nodo_2,
    length_km=0.0254, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15),  text_k=273.2, name="Tubo_Ida_nodo_1_nodo_2",k_mm=0.1
)
pipe_retorno_4= pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_2_ret, to_junction=j_nodo_1_ret,
    length_km=0.0254, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15),  text_k=273.2, name="Tubo_Retorno_nodo_1_nodo_2",k_mm=0.1
    
)
#1-3
pipe_ida_5 = pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_1, to_junction=j_nodo_3,
    length_km=0.0316, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15),  text_k=273.2, name="Tubo_Ida_nodo_1_nodo_3",k_mm=0.1
)
pipe_retorno_5= pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_3_ret, to_junction=j_nodo_1_ret,
   length_km=0.0316, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15),  text_k=273.2, name="Tubo_Retorno_nodo_1_nodo_3",k_mm=0.1  
)
#3-4
pipe_ida_6 = pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_3, to_junction=j_nodo_4,
    length_km=0.0498, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15),  text_k=273.2, name="Tubo_Ida_nodo_3_nodo_4",k_mm=0.1
)
pipe_retorno_6= pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_4_ret, to_junction=j_nodo_3_ret,
    length_km=0.0498, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15),  text_k=273.2, name="Tubo_Retorno_nodo_3_nodo_4",k_mm=0.1   
)
#4-5
pipe_ida_7 = pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_4, to_junction=j_nodo_5,
    length_km=0.0349, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15),  text_k=273.2, name="Tubo_Ida_nodo_4_nodo_5",k_mm=0.1
)
pipe_retorno_7= pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_5_ret, to_junction=j_nodo_4_ret,
    length_km=0.0349, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15),  text_k=273.2, name="Tubo_Retorno_nodo_4_nodo_5",k_mm=0.1   
)
#5-6
pipe_ida_8 = pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_5, to_junction=j_nodo_6,
    length_km=0.043, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15),  text_k=273.2, name="Tubo_Ida_nodo_5_nodo_6",k_mm=0.1
)
pipe_retorno_8= pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_6_ret, to_junction=j_nodo_5_ret,
    length_km=0.043, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15),  text_k=273.2, name="Tubo_Retorno_nodo_5_nodo_6",k_mm=0.1   
)
#Nodo 2
#Consumer1
pipe_ida_9 = pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_2, to_junction=j_cons_ida_1,
    length_km=0.0155, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15),  text_k=273.2, name="Tubo_Ida_nodo_2_consm_1",k_mm=0.1
)
pipe_retorno_9= pp.create_pipe_from_parameters(
    net, from_junction=j_cons_ret_1, to_junction=j_nodo_2_ret,
    length_km=0.0155, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15),  text_k=273.2, name="Tubo_Retorno_nodo_2_consm_1",k_mm=0.1  
)
#Consumer2
pipe_ida_10 = pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_2, to_junction=j_cons_ida_2,
    length_km=0.0122, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15),  text_k=273.2, name="Tubo_Ida_nodo_2_consm_2",k_mm=0.1
)
pipe_retorno_10= pp.create_pipe_from_parameters(
    net, from_junction=j_cons_ret_2, to_junction=j_nodo_2_ret,
    length_km=0.0122, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15),  text_k=273.2, name="Tubo_Retorno_nodo_2_consm_2",k_mm=0.1  
)
#Consumer3
pipe_ida_11 = pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_2, to_junction=j_cons_ida_3,
    length_km=0.0161, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15),  text_k=273.2, name="Tubo_Ida_nodo_2_consm_3",k_mm=0.1
)
pipe_retorno_11= pp.create_pipe_from_parameters(
    net, from_junction=j_cons_ret_3, to_junction=j_nodo_2_ret,
    length_km=0.0161, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15),  text_k=273.2, name="Tubo_Retorno_nodo_2_consm_3",k_mm=0.1 
)
#Nodo 4
#Consumer4
pipe_ida_12 = pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_4, to_junction=j_cons_ida_4,
    length_km=0.016, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15),  text_k=273.2, name="Tubo_Ida_nodo_4_consm_4",k_mm=0.1
)
pipe_retorno_12= pp.create_pipe_from_parameters(
    net, from_junction=j_cons_ret_4, to_junction=j_nodo_4_ret,
    length_km=0.016, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15),  text_k=273.2, name="Tubo_Retorno_nodo_4_consm_4",k_mm=0.1
)
#Consumer5
pipe_ida_13 = pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_4, to_junction=j_cons_ida_5,
    length_km=0.0158, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15),  text_k=273.2, name="Tubo_Ida_nodo_4_consm_5",k_mm=0.1
)
pipe_retorno_13= pp.create_pipe_from_parameters(
    net, from_junction=j_cons_ret_5, to_junction=j_nodo_4_ret,
    length_km=0.0158, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15),  text_k=273.2, name="Tubo_Retorno_nodo_4_consm_5",k_mm=0.1   
)
#Consumer6
pipe_ida_14 = pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_4, to_junction=j_cons_ida_6,
    length_km=0.0176, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15),  text_k=273.2, name="Tubo_Ida_nodo_4_consm_6",k_mm=0.1
)
pipe_retorno_14= pp.create_pipe_from_parameters(
    net, from_junction=j_cons_ret_6, to_junction=j_nodo_4_ret,
    length_km=0.0176, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15),  text_k=273.2, name="Tubo_Retorno_nodo_4_consm_6",k_mm=0.1  
)
#Nodo 6
#Consumer7
pipe_ida_15 = pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_6, to_junction=j_cons_ida_7,
    length_km=0.0336, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15),  text_k=273.2, name="Tubo_Ida_nodo_6_consm_7",k_mm=0.1
)
pipe_retorno_15= pp.create_pipe_from_parameters(
    net, from_junction=j_cons_ret_7, to_junction=j_nodo_6_ret,
    length_km=0.0336, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15),  text_k=273.2, name="Tubo_Retorno_nodo_6_consm_7",k_mm=0.1 
)
#Consumer8
pipe_ida_16 = pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_6, to_junction=j_cons_ida_8,
    length_km=0.0422, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15),  text_k=273.2, name="Tubo_Ida_nodo_6_consm_8",k_mm=0.1
)
pipe_retorno_16= pp.create_pipe_from_parameters(
    net, from_junction=j_cons_ret_8, to_junction=j_nodo_6_ret,
    length_km=0.0422, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15),  text_k=273.2, name="Tubo_Retorno_nodo_6_consm_8",k_mm=0.1
)
#Consumer9
pipe_ida_17 = pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_6, to_junction=j_cons_ida_9,
    length_km=0.0594, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15),  text_k=273.2, name="Tubo_Ida_nodo_6_consm_9",k_mm=0.1
)
pipe_retorno_17= pp.create_pipe_from_parameters(
    net, from_junction=j_cons_ret_9, to_junction=j_nodo_6_ret,
    length_km=0.0594, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15),  text_k=273.2, name="Tubo_Retorno_nodo_6_consm_9",k_mm=0.1
)

In [465]:
# =============================================================================
# COMPONENTE CONSUMIDOR (Potencia en W y Temp de Retorno)
# =============================================================================
# Conectamos el nudo de ida con el de retorno. 
# El sistema extraerá agua de la ida, absorberá los Watts indicados y la devolverá al retorno a la t_return_k.

#Consm1
pp.create_heat_consumer(
    net,
    from_junction=j_cons_ida_1 ,
    to_junction=j_cons_ret_1,
    qext_w=150000,
    treturn_k=305,
    name="Consumidor_1"
)
#Consm2
pp.create_heat_consumer(
    net,
    from_junction=j_cons_ida_2,
    to_junction=j_cons_ret_2,
    qext_w=150000,
    treturn_k=305,
    name="Consumidor_2"
)
#Consm3
pp.create_heat_consumer(
    net,
    from_junction=j_cons_ida_3,
    to_junction=j_cons_ret_3,
    qext_w=150000,
    treturn_k=305,
    name="Consumidor_3"
)
#Consm4
pp.create_heat_consumer(
    net,
    from_junction=j_cons_ida_4,
    to_junction=j_cons_ret_4,
    qext_w=150000,
    treturn_k=305,
    name="Consumidor_4"
)
#Consm5
pp.create_heat_consumer(
    net,
    from_junction=j_cons_ida_5,
    to_junction=j_cons_ret_5,
    qext_w=150000,
    treturn_k=305,
    name="Consumidor_5"
)
#Consm6
pp.create_heat_consumer(
    net,
    from_junction=j_cons_ida_6,
    to_junction=j_cons_ret_6,
    qext_w=150000,
    treturn_k=305,
    name="Consumidor_6"
)
#Consm7
pp.create_heat_consumer(
    net,
    from_junction=j_cons_ida_7,
    to_junction=j_cons_ret_7,
    qext_w=150000,
    treturn_k=305,
    name="Consumidor_7"
)
#Consm8
pp.create_heat_consumer(
    net,
    from_junction=j_cons_ida_8,
    to_junction=j_cons_ret_8,
    qext_w=150000,
    treturn_k=305,
    name="Consumidor_8"
)
#Consm9
pp.create_heat_consumer(
    net,
    from_junction=j_cons_ida_9,
    to_junction=j_cons_ret_9,
    qext_w=150000,
    treturn_k=305,
    name="Consumidor_9"
)


np.int64(8)

## Configuración de las plantas y funciones
Solo hay que tocar `PLANTS` (temperatura de ida y `q_max_w` de cada planta, en W) y, si quieres, la especificación de presión.

In [466]:
import numpy as np
import pandas as pd
import pandapipes as pp
from pandapipes.pf.pipeflow_setup import get_fluid

# ---------------------------------------------------------------------------
# CONFIGURACIÓN DE LAS PLANTAS (un solo sitio para tocar números)
# ---------------------------------------------------------------------------
P_REF_FLOW_BAR = 3.0      # presión de ida de la planta de referencia
P_REF_LIFT_BAR = 0.5      # plift -> presión de retorno = 2.5 bar
T_RETURN_GUESS = 305.0    # solo para el primer valor de mdot

PLANTS = [
    dict(name="Planta 1", t_supply_k=377.0, q_max_w=600_000),
    dict(name="Planta 2", t_supply_k=377.0, q_max_w=500_000),
    dict(name="Planta 3", t_supply_k=376.0, q_max_w=700_000),
]



In [467]:
def build_pumps(net, plants=PLANTS):
    """Cada planta i tiene DOS bombas con índice i:
       circ_pump_pressure[i] -> planta como SLACK (fija presión, potencia libre)
       circ_pump_mass[i]     -> planta con CAUDAL impuesto (potencia controlada)
       Se activa una u otra (o ninguna = planta apagada)."""
    flow_j = [j for j in net.junction.index if net.junction.name[j] in
              ("Fuente_Ida", "Fuente_2_Ida", "Fuente_3_Ida")]
    ret_j  = [j for j in net.junction.index if net.junction.name[j] in
              ("Fuente_Retorno", "Fuente_2_Retorno", "Fuente_3_Retorno")]
    for i, p in enumerate(plants):
        pp.create_circ_pump_const_pressure(
            net, return_junction=ret_j[i], flow_junction=flow_j[i],
            p_flow_bar=P_REF_FLOW_BAR, plift_bar=P_REF_LIFT_BAR,
            t_flow_k=p["t_supply_k"], type="pt", in_service=False,
            name=f"{p['name']}_slack")
    for i, p in enumerate(plants):
        pp.create_circ_pump_const_mass_flow(
            net, return_junction=ret_j[i], flow_junction=flow_j[i],
            mdot_flow_kg_per_s=0.1, t_flow_k=p["t_supply_k"],
            p_flow_bar=None, type="t", in_service=False,
            name=f"{p['name']}_caudal")

# ---------------------------------------------------------------------------
# POTENCIA DE UNA PLANTA  (misma definición que usabas: cp medio ida/retorno)
# ---------------------------------------------------------------------------
def plant_power(net, i, mode):
    res = net.res_circ_pump_pressure if mode == "slack" else net.res_circ_pump_mass
    r = res.loc[i]
    return r.qext_w

def mdot_for_power(net, plant, q_w, t_ret=T_RETURN_GUESS):
    fl = get_fluid(net)
    ts = plant["t_supply_k"]
    cp = 0.5 * (fl.get_heat_capacity(ts) + fl.get_heat_capacity(t_ret))
    return q_w / (cp * (ts - t_ret))

# ---------------------------------------------------------------------------
# RESOLVER UN ESTADO: 1 slack + plantas a potencia impuesta (+ 1 de equilibrio)
# ---------------------------------------------------------------------------
def solve_state(net, plants, slack, fixed, balance=None, tol_w=0.0001, max_it=30):
    """
    slack   : índice de la planta que fija la presión (bomba de presión)
    fixed   : {i: Q_objetivo_W} plantas con caudal impuesto que deben dar ese Q
    balance : (i_b, Q_slack_objetivo) -> planta i_b con caudal impuesto cuyo Q
              se ajusta hasta que la planta SLACK entregue exactamente Q_slack_objetivo
    """
    n = len(plants)
    flow_ctrl = dict(fixed)
    if balance is not None:
        flow_ctrl[balance[0]] = None          # objetivo dinámico
    # estado de las bombas
    reset(net,plants)
    for i in range(n):
        if i == slack: 
            net.circ_pump_pressure.at[i, "in_service"] =True
        if i in flow_ctrl:
            net.circ_pump_mass.at[i, "in_service"] =True
    net.circ_pump_pressure.at[slack, "p_flow_bar"] = P_REF_FLOW_BAR
    net.circ_pump_pressure.at[slack, "plift_bar"]  = P_REF_LIFT_BAR
    # semilla de caudales
    demanda = net.heat_consumer.qext_w.sum()
    for i, q in flow_ctrl.items():
        if q is None:
            q = max(demanda - sum(fixed.values()) - (balance[1] if balance else 0),
                    0.01 * plants[i]["q_max_w"])
        net.circ_pump_mass.at[i, "mdot_flow_kg_per_s"] = mdot_for_power(net, plants[i], q)

    for it in range(max_it):
        pp.pipeflow(net, mode="bidirectional")
        Q = {i: plant_power(net, i, "mass") for i in flow_ctrl}
        Q[slack] = plant_power(net, slack, "slack")
        err = {i: Q[i] - t for i, t in fixed.items()}
        if balance is not None:
            err[balance[0]] = Q[slack] - balance[1]
        if (not err) or max(abs(e) for e in err.values()) < tol_w:
            break
        for i, t in fixed.items():                       # Q impuesto
            net.circ_pump_mass.at[i, "mdot_flow_kg_per_s"] *= t / Q[i]
        if balance is not None:                          # Q de equilibrio
            b = balance[0]
            q_new = max(Q[b] + (Q[slack] - balance[1]), 1.0)
            net.circ_pump_mass.at[b, "mdot_flow_kg_per_s"] *= q_new / Q[b]
    return Q, it + 1

# ---------------------------------------------------------------------------
# DESPACHO POR ORDEN DE MÉRITO CON LÍMITES DE POTENCIA
# ---------------------------------------------------------------------------
def dispatch(net, plants=PLANTS, order=None, ref=None, tol_w=0.0001, verbose=True):
    """
    order : orden de prioridad (por defecto 0,1,2...)
    ref   : planta que da la ESPECIFICACIÓN DE PRESIÓN (por defecto la 1ª del orden).
            Si está saturada se queda a Q_max y la planta marginal ajusta su caudal
            hasta que ref entregue exactamente Q_max.
    """
    n = len(plants)
    order = list(range(n)) if order is None else list(order)
    print(order)
    ref = order[0] if ref is None else ref
    q_max = {i: plants[i]["q_max_w"] for i in range(n)}
    historial = []
    for k in range(n):
        active, sat, M = order[:k + 1], order[:k], order[k]
        slack = ref if ref in active else M
        print('This is slack', slack)
        fixed = {i: q_max[i] for i in sat if i != slack}
        print('Fixed:', fixed)
        balance = (M, q_max[slack]) if (slack in sat) else None
        Q, nit = solve_state(net, plants, slack, fixed, balance, tol_w)
        q_marg = Q[M]
        ok = q_marg <= q_max[M] + tol_w
        historial.append((k + 1, M, q_marg, ok))
        if verbose:
            print(f"Etapa {k+1}: plantas {[plants[i]['name'] for i in active]} "
                  f"| marginal={plants[M]['name']} da {q_marg/1e3:9.1f} kW "
                  f"(máx {q_max[M]/1e3:.1f}) -> {'OK' if ok else 'EXCEDE, se añade otra planta'}"
                  f"  [{nit} iter.]")
        if ok:
            return summary(net, plants, slack, Q, sat, M, feasible=True)
    deficit = q_marg - q_max[M]
    print(f"\n⚠ INFACTIBLE: las {n} plantas a su máximo no cubren la demanda. "
          f"Faltan {deficit/1e3:.1f} kW.")
    return summary(net, plants, slack, Q, sat, M, feasible=False, deficit=deficit)

def summary(net, plants, slack, Q, sat, M, feasible, deficit=0.0):
    rows = []
    for i, p in enumerate(plants):
        if i == slack:
            modo, r = "SLACK (presión)", net.res_circ_pump_pressure.loc[i]
        elif i in Q:
            modo, r = "caudal impuesto", net.res_circ_pump_mass.loc[i]
        else:
            rows.append(dict(Planta=p["name"], Modo="apagada", **{"Q [kW]": 0.0,
                "Q_max [kW]": p["q_max_w"]/1e3, "Carga [%]": 0.0, "mdot [kg/s]": 0.0,
                "T_ret [K]": np.nan, "p_ida [bar]": np.nan, "p_ret [bar]": np.nan}))
            continue
        rows.append(dict(Planta=p["name"], Modo=modo, **{"Q [kW]": Q[i]/1e3,
            "Q_max [kW]": p["q_max_w"]/1e3, "Carga [%]": 100*Q[i]/p["q_max_w"],
            "mdot [kg/s]": r.mdot_from_kg_per_s, "T_ret [K]": r.t_from_k,
            "p_ida [bar]": r.p_to_bar, "p_ret [bar]": r.p_from_bar}))
    df = pd.DataFrame(rows).set_index("Planta")
    q_plantas = sum(Q.values())
    q_cons = net.heat_consumer.qext_w.sum()
    return dict(tabla=df, feasible=feasible, deficit_w=deficit,
                q_plantas_w=q_plantas, q_consumidores_w=q_cons,
                perdidas_w=q_plantas - q_cons)
def reset(net, plants):
    """Apaga todas las plantas y resetea los caudales de las bombas."""
    for i in range(len(plants)):
        net.circ_pump_pressure.at[i, "in_service"] = False
        net.circ_pump_mass.at[i, "in_service"] = False


## Simulación
Las bombas de la celda de condiciones de contorno antigua se sustituyen por `build_pumps` (2 por planta, todas apagadas hasta que `dispatch` decide).

In [468]:
build_pumps(net)
res = dispatch(net)   # ref = planta 1, orden 1->2->3
res['tabla']

[0, 1, 2]
This is slack 0
Fixed: {}
Etapa 1: plantas ['Planta 1'] | marginal=Planta 1 da    1371.3 kW (máx 600.0) -> EXCEDE, se añade otra planta  [1 iter.]
This is slack 0
Fixed: {}
Etapa 2: plantas ['Planta 1', 'Planta 2'] | marginal=Planta 2 da     772.6 kW (máx 500.0) -> EXCEDE, se añade otra planta  [4 iter.]
This is slack 0
Fixed: {1: 500000}
Etapa 3: plantas ['Planta 1', 'Planta 2', 'Planta 3'] | marginal=Planta 3 da     275.5 kW (máx 700.0) -> OK  [7 iter.]


,Modo,Q [kW],Q_max [kW],Carga [%],mdot [kg/s],T_ret [K],p_ida [bar],p_ret [bar]
Planta,,,,,,,,
Planta 1,SLACK (presión),600.000000,600.0,100.000000,1.977825,304.745417,3.000000,2.500000
Planta 2,caudal impuesto,500.000000,500.0,100.000000,1.646454,304.669365,2.999827,2.500184
Planta 3,caudal impuesto,275.461332,700.0,39.351619,0.918827,304.587888,2.999251,2.500795


In [469]:
print(f"ΣQ plantas      = {res['q_plantas_w']/1e3:8.2f} kW")
print(f"Demanda consum. = {res['q_consumidores_w']/1e3:8.2f} kW")
print(f"Pérdidas red    ≈ {res['perdidas_w']/1e3:8.2f} kW   (cambian con el reparto de caudales)")
print('Factible:', res['feasible'])

ΣQ plantas      =  1375.46 kW
Demanda consum. =  1350.00 kW
Pérdidas red    ≈    25.46 kW   (cambian con el reparto de caudales)
Factible: True


## Otros casos
**Tus `Q_max` originales** (≈137 kW por planta): 3 × 137 kW ≈ 412 kW, pero la demanda es 1350 kW + pérdidas. Es imposible; en tu versión la planta 3 acababa dando 1101 kW con un límite de 137.8 kW. Ahora se detecta.

In [470]:
for p, q in zip(PLANTS, [137135, 137130, 137777]):
    p['q_max_w'] = q
reset(net, PLANTS)
res = dispatch(net)
res['tabla']

[0, 1, 2]
This is slack 0
Fixed: {}
Etapa 1: plantas ['Planta 1'] | marginal=Planta 1 da    1371.3 kW (máx 137.1) -> EXCEDE, se añade otra planta  [1 iter.]
This is slack 0
Fixed: {}
Etapa 2: plantas ['Planta 1', 'Planta 2'] | marginal=Planta 2 da    1235.5 kW (máx 137.1) -> EXCEDE, se añade otra planta  [5 iter.]
This is slack 0
Fixed: {1: 137130}
Etapa 3: plantas ['Planta 1', 'Planta 2', 'Planta 3'] | marginal=Planta 3 da    1101.1 kW (máx 137.8) -> EXCEDE, se añade otra planta  [7 iter.]

⚠ INFACTIBLE: las 3 plantas a su máximo no cubren la demanda. Faltan 963.4 kW.


,Modo,Q [kW],Q_max [kW],Carga [%],mdot [kg/s],T_ret [K],p_ida [bar],p_ret [bar]
Planta,,,,,,,,
Planta 1,SLACK (presión),137.135000,137.135,100.000000,0.451518,304.660593,3.000000,2.500000
Planta 2,caudal impuesto,137.130000,137.130,100.000000,0.451415,304.646741,3.000075,2.499916
Planta 3,caudal impuesto,1101.130764,137.777,799.212324,3.678534,304.696720,3.002592,2.497328


In [471]:
net.circ_pump_pressure

,name,return_junction,flow_junction,p_flow_bar,t_flow_k,plift_bar,in_service,type
0,Planta 1_slack,18,0,3.0,377.0,0.5,True,pt
1,Planta 2_slack,19,1,3.0,377.0,0.5,False,pt
2,Planta 3_slack,20,2,3.0,376.0,0.5,False,pt


In [472]:
net.circ_pump_mass

,name,return_junction,flow_junction,p_flow_bar,t_flow_k,mdot_flow_kg_per_s,in_service,type
0,Planta 1_caudal,18,0,NaN,377.0,0.100000,False,t
1,Planta 2_caudal,19,1,NaN,377.0,0.451415,True,t
2,Planta 3_caudal,20,2,NaN,376.0,3.678534,True,t


**Misma red, otros límites, y otra planta como referencia de presión** (el reparto de potencias no cambia, solo el nivel de presión).

In [473]:
for p, q in zip(PLANTS, [600_000, 500_000, 700_000]):
    p['q_max_w'] = q
reset(net,PLANTS)
res_a = dispatch(net, ref=0, verbose=True)
reset(net,PLANTS)
print("Second Stage")
res_b = dispatch(net, ref=2, verbose=True)
print('ref = Planta 1'); display(res_a['tabla'][['Modo','Q [kW]','Carga [%]','p_ida [bar]']].round(3))
print('ref = Planta 3'); display(res_b['tabla'][['Modo','Q [kW]','Carga [%]','p_ida [bar]']].round(3))

[0, 1, 2]
This is slack 0
Fixed: {}
Etapa 1: plantas ['Planta 1'] | marginal=Planta 1 da    1371.3 kW (máx 600.0) -> EXCEDE, se añade otra planta  [1 iter.]
This is slack 0
Fixed: {}
Etapa 2: plantas ['Planta 1', 'Planta 2'] | marginal=Planta 2 da     772.6 kW (máx 500.0) -> EXCEDE, se añade otra planta  [4 iter.]
This is slack 0
Fixed: {1: 500000}
Etapa 3: plantas ['Planta 1', 'Planta 2', 'Planta 3'] | marginal=Planta 3 da     275.5 kW (máx 700.0) -> OK  [7 iter.]
Second Stage
[0, 1, 2]
This is slack 0
Fixed: {}
Etapa 1: plantas ['Planta 1'] | marginal=Planta 1 da    1371.3 kW (máx 600.0) -> EXCEDE, se añade otra planta  [1 iter.]
This is slack 1
Fixed: {0: 600000}
Etapa 2: plantas ['Planta 1', 'Planta 2'] | marginal=Planta 2 da     772.6 kW (máx 500.0) -> EXCEDE, se añade otra planta  [4 iter.]
This is slack 2
Fixed: {0: 600000, 1: 500000}
Etapa 3: plantas ['Planta 1', 'Planta 2', 'Planta 3'] | marginal=Planta 3 da     275.5 kW (máx 700.0) -> OK  [4 iter.]
ref = Planta 1


,Modo,Q [kW],Carga [%],p_ida [bar]
Planta,,,,
Planta 1,SLACK (presión),600.000,100.000,3.000
Planta 2,caudal impuesto,500.000,100.000,3.000
Planta 3,caudal impuesto,275.461,39.352,2.999


ref = Planta 3


,Modo,Q [kW],Carga [%],p_ida [bar]
Planta,,,,
Planta 1,caudal impuesto,600.000,100.000,3.001
Planta 2,caudal impuesto,500.000,100.000,3.001
Planta 3,SLACK (presión),275.461,39.352,3.000


**Otro orden de prioridad** (por ejemplo 3 → 2 → 1, con la planta 1 como referencia de presión):

In [474]:
reset(net, PLANTS)
res = dispatch(net, order=[2, 1, 0], ref=0)
res['tabla'].round

[2, 1, 0]
This is slack 2
Fixed: {}
Etapa 1: plantas ['Planta 3'] | marginal=Planta 3 da    1372.4 kW (máx 700.0) -> EXCEDE, se añade otra planta  [1 iter.]
This is slack 1
Fixed: {2: 700000}
Etapa 2: plantas ['Planta 3', 'Planta 2'] | marginal=Planta 2 da     673.7 kW (máx 500.0) -> EXCEDE, se añade otra planta  [5 iter.]
This is slack 0
Fixed: {2: 700000, 1: 500000}
Etapa 3: plantas ['Planta 3', 'Planta 2', 'Planta 1'] | marginal=Planta 1 da     175.4 kW (máx 600.0) -> OK  [5 iter.]


<bound method DataFrame.round of                      Modo      Q [kW]  Q_max [kW]   Carga [%]  mdot [kg/s]  \
Planta                                                                       
Planta 1  SLACK (presión)  175.447649       600.0   29.241275     0.578032   
Planta 2  caudal impuesto  500.000000       500.0  100.000000     1.647615   
Planta 3  caudal impuesto  700.000000       700.0  100.000000     2.337222   

           T_ret [K]  p_ida [bar]  p_ret [bar]  
Planta                                          
Planta 1  304.706812     3.000000     2.500000  
Planta 2  304.720315     3.000184     2.499802  
Planta 3  304.658303     3.000798     2.499160  >

In [475]:
net.circ_pump_pressure

,name,return_junction,flow_junction,p_flow_bar,t_flow_k,plift_bar,in_service,type
0,Planta 1_slack,18,0,3.0,377.0,0.5,True,pt
1,Planta 2_slack,19,1,3.0,377.0,0.5,False,pt
2,Planta 3_slack,20,2,3.0,376.0,0.5,False,pt


In [476]:
for p, q in zip(PLANTS, [137135, 137130, 6000000000]):
     p['q_max_w'] = q
reset(net, PLANTS)
res = dispatch(net, order=[2, 1, 0], ref=0)
res['tabla'].round

[2, 1, 0]
This is slack 2
Fixed: {}
Etapa 1: plantas ['Planta 3'] | marginal=Planta 3 da    1372.4 kW (máx 6000000.0) -> OK  [1 iter.]


<bound method DataFrame.round of                      Modo       Q [kW]   Q_max [kW]  Carga [%]  mdot [kg/s]  \
Planta                                                                        
Planta 1          apagada     0.000000      137.135   0.000000     0.000000   
Planta 2          apagada     0.000000      137.130   0.000000     0.000000   
Planta 3  SLACK (presión)  1372.369102  6000000.000   0.022873     4.586492   

           T_ret [K]  p_ida [bar]  p_ret [bar]  
Planta                                          
Planta 1         NaN          NaN          NaN  
Planta 2         NaN          NaN          NaN  
Planta 3  304.725241          3.0          2.5  >

In [477]:
net.res_circ_pump_pressure

,p_from_bar,p_to_bar,t_from_k,t_to_k,t_outlet_k,mdot_from_kg_per_s,mdot_to_kg_per_s,vdot_m3_per_s,deltat_k,qext_w
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,NaN
2,2.5,3.0,304.725241,376.0,376.0,4.586492,-4.586492,0.0047,-71.274759,1.372369e+06
